In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-ridho-model'  # ckpt = 4000
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-14'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # min_ankle_height 弊害
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
env_cfg["episode_length_s"] = 60.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
env_cfg['dt'] = 0.01
env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 60.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 100.0]}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[ 0.0631, -0.3333,  0.6196, -0.3140, -1.1508,  0.0816, -0.0783,  0.7042,
         -0.1964,  0.4932, -1.0956,  0.0864]], device='cuda:0')
Scaled actions :  tensor([[ 0.0631, -0.3333,  0.6196, -0.3140, -1.1508,  0.0816, -0.0783,  0.7042,
         -0.1964,  0.4932, -1.0956,  0.0864]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0845e-10,  3.7697e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -8.5621e-08,
         -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,  7.6633e-07,
          1.4362e-08,  3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04,
         -3.4389e-08, -4.2811e-06, -2.3818e-06, -7.0895e-03,  1.8232e-02,
         -9.5524e-03,  3.8316e-05,  7.1809e-07,  1.5422e-06, -7.0859e-03,
          1.8227e-02, -9.5517e-03, -1.7194e-06,  6.3118e-02, -3.3332e-01,
          6.1957e-01, -3.1398e-01, -1.1508e+00,  8.1580e-02, -7.8320e-02,
          7.0416e-01, -1.9644e-01,  4.9324e-01, -1.0956e+00,  8.6366e-02]],
       device='cuda:0')
torques: [-1.58794924e-16 -1.00915112e-15  2.37314235e-06  7.56007923e-06
  1.59905156e-06  6.12617365e-17 -6.48048861e-18 -7.17891685e-16
  2.37314235e-06  7.56007923e-06  1.59905156e-06 -8.75046755e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.0213, -0.4133,  0.1727, -0.2547, -1.8872, -0.1760,  0.3333,  0.3102,
         -0.6728,  0.6266, -1.5611, -0.1151]], device='cuda:0')
Scaled actions :  tensor([[-0.0213, -0.4133,  0.1727, -0.2547, -1.8872, -0.1760,  0.3333,  0.3102,
         -0.6728,  0.6266, -1.5611, -0.1151]], device='cuda:0')
obs :  tensor([[-0.1909, -0.1155,  0.1177, -0.0027,  0.0041, -1.0000,  1.0000,  0.0000,
          0.0000,  0.0203,  0.0033,  0.0071,  0.0171, -0.0852,  0.0443, -0.0225,
          0.0132, -0.0173,  0.0379, -0.1082,  0.0407,  0.1891,  0.0302,  0.0706,
          0.1411, -0.7625,  0.3995, -0.1333,  0.1162, -0.1475,  0.3238, -0.9733,
          0.3959, -0.0213, -0.4133,  0.1727, -0.2547, -1.8872, -0.1760,  0.3333,
          0.3102, -0.6728,  0.6266, -1.5611, -0.1151]], device='cuda:0')
torques: [  13.09048141  200.         -200.          200.         -200.
  120.75465141  100.09685291 -200.          200.         -200.
 -200.          105.51386798]
データ収集: step 3


In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[-0.4472, -0.7312,  0.3839, -0.5572, -0.5492, -0.9948,  0.6567, -0.2836,
          0.6768, -0.4194,  0.0766,  0.1844]], device='cuda:0')
Scaled actions :  tensor([[-0.4472, -0.7312,  0.3839, -0.5572, -0.5492, -0.9948,  0.6567, -0.2836,
          0.6768, -0.4194,  0.0766,  0.1844]], device='cuda:0')
obs :  tensor([[-3.7718e-01, -9.4164e-02,  7.3400e-02, -7.1304e-03,  1.5808e-02,
         -9.9985e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.3895e-02,
          1.7887e-02,  3.6580e-02,  3.3304e-02, -3.3365e-01,  2.1636e-02,
         -8.7018e-04,  4.7739e-02, -6.5951e-02,  1.3309e-01, -4.0821e-01,
          2.0576e-02, -8.5480e-02,  1.0178e-01,  2.0714e-01,  2.8040e-02,
         -1.6395e+00, -4.6790e-01,  3.1175e-01,  2.1743e-01, -3.2814e-01,
          5.9140e-01, -1.9292e+00, -3.0080e-01, -4.4724e-01, -7.3122e-01,
          3.8388e-01, -5.5720e-01, -5.4923e-01, -9.9479e-01,  6.5669e-01,
         -2.8358e-01,  6.7683e-01, -4.1937e-01,  7.6611e-02,  1.8

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[ 0.0598, -0.6702, -1.0758, -0.9405,  0.8005, -0.2785, -0.0999, -0.0212,
         -0.1805, -0.4779,  0.6373,  0.1861]], device='cuda:0')
Scaled actions :  tensor([[ 0.0598, -0.6702, -1.0758, -0.9405,  0.8005, -0.2785, -0.0999, -0.0212,
         -0.1805, -0.4779,  0.6373,  0.1861]], device='cuda:0')
obs :  tensor([[ 0.1832, -0.6396,  0.0082, -0.0242,  0.0188, -0.9995,  1.0000,  0.0000,
          0.0000, -0.0561,  0.0233,  0.1090,  0.0415, -0.6419, -0.1395,  0.1121,
          0.0594, -0.0936,  0.2119, -0.6892,  0.0483, -0.5657, -0.0681,  0.4558,
          0.0321, -1.3465, -0.9360,  0.6917, -0.0566,  0.0073,  0.2433, -0.9772,
          0.2152,  0.0598, -0.6702, -1.0758, -0.9405,  0.8005, -0.2785, -0.0999,
         -0.0212, -0.1805, -0.4779,  0.6373,  0.1861]], device='cuda:0')
torques: [-200. -200.  200. -200.  200. -200.  200. -200. -200. -200.  200.  200.]
データ収集: step 5


In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[ 0.9183,  0.9574, -0.5954, -1.7813,  1.3398,  0.5091, -0.9412,  1.1045,
         -0.7797, -0.3873,  0.4913,  0.3189]], device='cuda:0')
Scaled actions :  tensor([[ 0.9183,  0.9574, -0.5954, -1.7813,  1.3398,  0.5091, -0.9412,  1.1045,
         -0.7797, -0.3873,  0.4913,  0.3189]], device='cuda:0')
obs :  tensor([[ 0.5308, -0.1097,  0.0071, -0.0365,  0.0050, -0.9993,  1.0000,  0.0000,
          0.0000, -0.1186, -0.0125,  0.1728,  0.0394, -0.8070, -0.2619,  0.1998,
          0.0349, -0.1049,  0.2412, -0.7773,  0.1043, -0.1109, -0.2374,  0.2135,
         -0.0290, -0.4205, -0.3339,  0.2221, -0.1468, -0.0806,  0.0617, -0.0042,
          0.0857,  0.9183,  0.9574, -0.5954, -1.7813,  1.3398,  0.5091, -0.9412,
          1.1045, -0.7797, -0.3873,  0.4913,  0.3189]], device='cuda:0')
torques: [-200.          194.60220776   18.51851207 -200.          200.
 -200.          200.         -200.         -200.         -200.
  200.          200.        ]
データ収集: step 6


In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[-0.5664,  0.4800,  0.4841, -0.8557,  0.9559,  0.6401,  0.0396, -0.0426,
          1.0239,  0.2437, -0.0925,  0.2104]], device='cuda:0')
Scaled actions :  tensor([[-0.5664,  0.4800,  0.4841, -0.8557,  0.9559,  0.6401,  0.0396, -0.0426,
          1.0239,  0.2437, -0.0925,  0.2104]], device='cuda:0')
obs :  tensor([[-0.0840,  0.5224,  0.0511, -0.0275, -0.0047, -0.9996,  1.0000,  0.0000,
          0.0000, -0.0834, -0.0262,  0.1914,  0.0273, -0.7955, -0.2171,  0.1923,
          0.0351, -0.1480,  0.2403, -0.6707,  0.1655,  0.4151,  0.0664, -0.0022,
         -0.0899,  0.4535,  0.6711, -0.2525,  0.1327, -0.3245, -0.0624,  0.9731,
          0.2803, -0.5664,  0.4800,  0.4841, -0.8557,  0.9559,  0.6401,  0.0396,
         -0.0426,  1.0239,  0.2437, -0.0925,  0.2104]], device='cuda:0')
torques: [-200.        200.       -200.       -200.        200.         67.806234
  200.        200.       -200.       -200.        200.        200.      ]
データ収集: step 7


In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[-0.3537, -0.2701,  0.3563, -0.3340, -0.4680,  0.5029,  0.5177, -0.5117,
          1.4503,  0.7352, -0.9514, -0.1507]], device='cuda:0')
Scaled actions :  tensor([[-0.3537, -0.2701,  0.3563, -0.3340, -0.4680,  0.5029,  0.5177, -0.5117,
          1.4503,  0.7352, -0.9514, -0.1507]], device='cuda:0')
obs :  tensor([[-1.4477e-01, -8.5753e-02,  5.3766e-01, -1.9993e-02,  2.8289e-04,
         -9.9980e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -5.2955e-02,
         -1.1967e-02,  2.2251e-01, -1.3194e-02, -6.1260e-01, -5.4290e-03,
          1.4460e-01,  5.8732e-02, -1.8542e-01,  2.2484e-01, -4.5853e-01,
          1.9334e-01, -6.4045e-02,  7.7176e-02,  3.0460e-01, -3.2915e-01,
          1.3185e+00,  1.4112e+00, -2.5407e-01,  1.0136e-01, -8.1543e-02,
         -7.4908e-02,  1.0887e+00,  1.0252e-01, -3.5371e-01, -2.7010e-01,
          3.5630e-01, -3.3397e-01, -4.6801e-01,  5.0291e-01,  5.1770e-01,
         -5.1171e-01,  1.4503e+00,  7.3515e-01, -9.5138e-01, -1.5

In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 0.5070,  0.1511, -1.2428, -0.1354, -0.1192,  0.1057,  0.4398,  0.4766,
         -0.3132,  0.2088, -0.7805, -0.2398]], device='cuda:0')
Scaled actions :  tensor([[ 0.5070,  0.1511, -1.2428, -0.1354, -0.1192,  0.1057,  0.4398,  0.4766,
         -0.3132,  0.2088, -0.7805, -0.2398]], device='cuda:0')
obs :  tensor([[ 0.3074, -0.7317,  0.6670, -0.0385, -0.0031, -0.9993,  1.0000,  0.0000,
          0.0000, -0.1012, -0.0393,  0.2754, -0.0294, -0.3979,  0.2259,  0.1420,
          0.0502, -0.1736,  0.2194, -0.3491,  0.1120, -0.4379, -0.2982,  0.3208,
         -0.0676,  0.9762,  1.3672,  0.1918, -0.1572,  0.1632,  0.0155,  0.1034,
         -0.8111,  0.5070,  0.1511, -1.2428, -0.1354, -0.1192,  0.1057,  0.4398,
          0.4766, -0.3132,  0.2088, -0.7805, -0.2398]], device='cuda:0')
torques: [ 200.         -200.          200.          200.         -200.
 -200.         -200.         -200.          186.5264884  -200.
  -60.69261235  200.        ]
データ収集: step 9


In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[ 0.4441,  0.8652, -1.2059, -0.9720, -0.9456, -0.3247, -0.2572,  0.7478,
         -0.4751, -0.5118, -0.1447,  0.6485]], device='cuda:0')
Scaled actions :  tensor([[ 0.4441,  0.8652, -1.2059, -0.9720, -0.9456, -0.3247, -0.2572,  0.7478,
         -0.4751, -0.5118, -0.1447,  0.6485]], device='cuda:0')
obs :  tensor([[-0.3766, -0.1236,  0.1339, -0.0535, -0.0013, -0.9986,  1.0000,  0.0000,
          0.0000, -0.1227, -0.0590,  0.3158, -0.0492, -0.1869,  0.3906,  0.2234,
          0.0454, -0.1706,  0.2269, -0.4293, -0.0769,  0.1605,  0.0714,  0.0249,
          0.0469,  0.2304,  0.3023,  0.4676,  0.0804, -0.1013,  0.0368, -0.7147,
         -0.6320,  0.4441,  0.8652, -1.2059, -0.9720, -0.9456, -0.3247, -0.2572,
          0.7478, -0.4751, -0.5118, -0.1447,  0.6485]], device='cuda:0')
torques: [ -15.64385866  200.         -200.          -79.74431788  -27.51407761
  200.          200.          200.         -200.          -54.05906739
 -200.         -200.        

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[-0.3560, -1.5095,  0.4851,  0.3401, -0.1180,  0.1126, -1.3499, -1.4424,
          1.5594,  0.6042,  0.6861,  0.2091]], device='cuda:0')
Scaled actions :  tensor([[-0.3560, -1.5095,  0.4851,  0.3401, -0.1180,  0.1126, -1.3499, -1.4424,
          1.5594,  0.6042,  0.6861,  0.2091]], device='cuda:0')
obs :  tensor([[-0.9693,  0.4811,  0.1966, -0.0454,  0.0285, -0.9986,  1.0000,  0.0000,
          0.0000, -0.0327, -0.0050,  0.3059, -0.0565, -0.2458,  0.3454,  0.2658,
          0.0886, -0.2181,  0.2192, -0.4639, -0.1003,  0.6933,  0.4248, -0.1239,
         -0.1084, -0.7230, -0.6595,  0.0050,  0.3317, -0.3408, -0.1034,  0.2694,
          0.3045, -0.3560, -1.5095,  0.4851,  0.3401, -0.1180,  0.1126, -1.3499,
         -1.4424,  1.5594,  0.6042,  0.6861,  0.2091]], device='cuda:0')
torques: [-200.  200. -200. -200.  200.  200.  200.  200. -200. -200. -200. -200.]
データ収集: step 11


In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[-0.3857, -1.0424,  0.3809, -0.7239,  0.0742,  0.3637, -0.0529, -0.3702,
          0.8767,  0.4255, -0.2487, -0.1189]], device='cuda:0')
Scaled actions :  tensor([[-0.3857, -1.0424,  0.3809, -0.7239,  0.0742,  0.3637, -0.0529, -0.3702,
          0.8767,  0.4255, -0.2487, -0.1189]], device='cuda:0')
obs :  tensor([[-2.1940e-01, -1.0858e-01,  8.6858e-01, -3.8291e-02,  5.1810e-02,
         -9.9792e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  3.6942e-02,
          3.5791e-02,  2.9492e-01, -6.0511e-02, -2.8483e-01,  2.6150e-01,
          2.1115e-01,  1.3365e-01, -2.5810e-01,  2.1051e-01, -3.0490e-01,
          2.5891e-04,  5.7576e-02,  3.4180e-02,  1.7001e-02,  5.8092e-02,
          2.3543e-01, -3.2841e-01, -5.0554e-01,  1.4410e-01, -8.4209e-02,
          8.2611e-03,  1.2249e+00,  4.5807e-01, -3.8568e-01, -1.0424e+00,
          3.8088e-01, -7.2391e-01,  7.4207e-02,  3.6368e-01, -5.2913e-02,
         -3.7022e-01,  8.7669e-01,  4.2546e-01, -2.4875e-01, -1.

In [36]:
# 既存のforループを置き換え
num_steps = 100
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=0.520, Scaled action max=0.520
Step 1/100, Total steps: 1212
steps: 1212
actions : tensor([[-0.4669, -0.3272,  0.1930, -1.0904, -0.2743, -0.4342,  0.4959, -0.4034,
          0.5200,  0.1078,  0.0353,  0.2987]], device='cuda:0')
target_dof_pos: tensor([[ 0.0235,  0.7089, -0.1583,  0.6532, -1.5833, -0.8982, -0.2654,  1.1812,
          0.5198,  2.6944, -0.2515,  1.1326]], device='cuda:0')
Step 1: Original action max=1.152, Scaled action max=1.152
Step 2: Original action max=1.524, Scaled action max=1.524
Step 21/100, Total steps: 1232
steps: 1232
actions : tensor([[-0.4359, -0.3440, -0.9936, -0.9588, -0.7379,  0.9131, -0.4658, -0.3444,
         -0.4043, -0.4098, -0.3517, -0.0059]], device='cuda:0')
target_dof_pos: tensor([[ 0.3762,  0.9242,  0.0971,  1.3861, -0.2404,  0.0436,  0.5319, -0.0601,
         -0.4653,  2.0881, -0.3877,  0.6263]], device='cuda:0')
Step 41/100, Total steps: 1252
steps: 1252
actions : tensor([[-0.4682, -0.4293, -0.2300,  0.7093, -0.2157,

In [25]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [75]:
env.sim.stop()

In [50]:
env.reset()
cnt = 0

In [61]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-terrain2-kp2000kd50-kpkdrand-14_ckpt100_scale1.0_rotorInertia0.1.csv
データ形状: (362, 58)
